In [2]:
import numpy as np
import cvxpy as cp
import plotly.graph_objects as go
from scipy.spatial import ConvexHull

In [14]:
#3 Finger Symmetric

# Step 1: Define the gripper and object (3D)
contact1_pos = np.array([1, 1, 0])   # Finger 1 (right face)
contact2_pos = np.array([-1, 1, 0])  # Finger 2 (left face)
contact3_pos = np.array([0, -1, 0])   # Thumb (top face)
mu = 0.5  # Friction coefficient

alpha = np.pi/6
beta = np.pi/2 - alpha
# Step 2: Define wrench basis for each contact
normal1 = np.array([-1 * np.cos(alpha), -1 * np.sin(alpha), 0], dtype=float)  # Right face, inward
normal2 = np.array([1 * np.cos(alpha), -1 * np.cos(alpha), 0], dtype=float)   # Left face, inward
normal3 = np.array([0, 1, 0], dtype=float)  # Top face, inward

normal1 /= np.linalg.norm(normal1)
normal2 /= np.linalg.norm(normal2)
normal3 /= np.linalg.norm(normal3)


# Contact 1 
tangent1_a = np.array([0, 0, 1])
tangent1_b = np.array([1 * np.cos(beta), -1 * np.sin(beta), 0])

# Contact 2 
tangent2_a = np.array([0, 0, 1])
tangent2_b = np.array([1 * np.cos(beta), 1 * np.sin(beta), 0])

# Contact 3 
tangent3_a = np.array([0, 0, 1])
tangent3_b = np.array([-1, 0, 0])

f1_n = normal1
f1_t1 = normal1 + mu * tangent1_a
f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2
f2_t1 = normal2 + mu * tangent2_a
f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3
f3_t1 = normal3 + mu * tangent3_a
f3_t2 = normal3 + mu * tangent3_b


f1_t1 /= np.linalg.norm(f1_t1)
f1_t2 /= np.linalg.norm(f1_t2)
f2_t1 /= np.linalg.norm(f2_t1)
f2_t2 /= np.linalg.norm(f2_t2)
f3_t1 /= np.linalg.norm(f3_t1)
f3_t2 /= np.linalg.norm(f3_t2)


def wrench(pos, force):
    r = pos
    f = force
    torque = np.cross(r, f)
    return np.concatenate([f, torque])

w1_n = wrench(contact1_pos, f1_n)
w1_t1 = wrench(contact1_pos, f1_t1)
w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n)
w2_t1 = wrench(contact2_pos, f2_t1)
w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n)
w3_t1 = wrench(contact3_pos, f3_t1)
w3_t2 = wrench(contact3_pos, f3_t2)



G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2])


# Step 3: Compute Grasp Wrench Space vertices
vertices = []
for coeffs in np.eye(9):
    vertices.append(G @ coeffs)
for coeffs in [np.ones(9), -np.ones(9)]:
    vertices.append(G @ coeffs)
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Step 4: Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(9)

constraints = [
    w == G @ f,
    cp.norm(f, 1) <= 1,
    f >= 0
]

if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)

objective = cp.Maximize(r)
problem = cp.Problem(objective, constraints)
problem.solve()

quality = problem.value
print(f"Grasp Quality (Largest Minimum Wrench): {quality:.4f} N or Nm")
print(f"Center of largest inscribed ball: {w.value}")

# Step 5: Interactive 3D Plot with Plotly (Fx, Fy, Tz projection)
fig = go.Figure()


# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', marker=dict(size=8, color='red'),
    name='Origin'
))

# Center of the ball

fig.add_trace(go.Scatter3d(
    x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
    mode='markers', marker=dict(size=8, color='green'),
    name='Center'
))


# Quality sphere (parametric)
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x = quality * np.cos(u) * np.sin(v) + w.value[0]
y = quality * np.sin(u) * np.sin(v) + w.value[1]
z = quality * np.cos(v) + w.value[5]
fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    opacity=0.4,  # Slightly more visible but still see-through
    colorscale='Blues',
    showscale=False,
    name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
    surfacecolor=np.ones_like(x),  # Uniform red
    contours=dict(
        x=dict(show=True, color='powderblue', width=1),
        y=dict(show=True, color='powderblue', width=1),
        z=dict(show=True, color='powderblue', width=1)
    )  # Add wireframe-like contours
))

fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],  # Dummy point, won't be visible
    mode='markers',
    marker=dict(size=8, color='powderblue'),
    name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)'
))

# Layout
fig.update_layout(
    title='Grasp Wrench Space (Fx-Fy-Tz Projection) / 3 Finger Symmetric',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

Grasp Quality (Largest Minimum Wrench): 0.1550 N or Nm
Center of largest inscribed ball: [-0.0901226  -0.08613962  0.2316219   0.08489313 -0.00816167 -0.00682088]


In [ ]:
# Four Finger Symmetric
# Step 1: Define the gripper and object (3D)
contact1_pos = np.array([1, 1, 0])   # Finger 1 (right face)
contact2_pos = np.array([-1, 1, 0])  # Finger 2 (left face)
contact3_pos = np.array([-1, -1, 0])   # Thumb (top face)
contact4_pos = np.array([1, -1, 0])
mu = 0.5  # Friction coefficient

# Step 2: Define wrench basis for each contact
normal1 = np.array([-1, -1, 0], dtype = float)  # Right face, inward
normal2 = np.array([1, -1, 0], dtype = float)   # Left face, inward
normal3 = np.array([1, 1, 0], dtype = float)  # Top face, inward
normal4 = np.array([-1, 1, 0], dtype = float)

normal1 /= np.linalg.norm(normal1)
normal2 /= np.linalg.norm(normal2)
normal3 /= np.linalg.norm(normal3)
normal4 /= np.linalg.norm(normal4) 

# Contact 1 (Finger 1)
tangent1_a = np.array([0, 0, 1])
tangent1_b = np.array([1, -1, 0])

# Contact 2 (Finger 2)
tangent2_a = np.array([0, 0, 1])
tangent2_b = np.array([1, 1, 0])

# Contact 3 (Thumb)
tangent3_a = np.array([0, 0, 1])
tangent3_b = np.array([-1, 1, 0])

# Contact 4
tangent4_a = np.array([0, 0, 1])
tangent4_b = np.array([-1, -1, 0])
f1_n = normal1
f1_t1 = normal1 + mu * tangent1_a
f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2
f2_t1 = normal2 + mu * tangent2_a
f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3
f3_t1 = normal3 + mu * tangent3_a
f3_t2 = normal3 + mu * tangent3_b
f4_n = normal4
f4_t1 = normal4 + mu * tangent4_a
f4_t2 = normal4 + mu * tangent4_b



f1_t1 /= np.linalg.norm(f1_t1)
f1_t2 /= np.linalg.norm(f1_t2)
f2_t1 /= np.linalg.norm(f2_t1)
f2_t2 /= np.linalg.norm(f2_t2)
f3_t1 /= np.linalg.norm(f3_t1)
f3_t2 /= np.linalg.norm(f3_t2)
f4_t1 /= np.linalg.norm(f4_t1)
f4_t2 /= np.linalg.norm(f4_t2)


def wrench(pos, force):
    r = pos
    f = force
    torque = np.cross(r, f)
    return np.concatenate([f, torque])

w1_n = wrench(contact1_pos, f1_n)
w1_t1 = wrench(contact1_pos, f1_t1)
w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n)
w2_t1 = wrench(contact2_pos, f2_t1)
w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n)
w3_t1 = wrench(contact3_pos, f3_t1)
w3_t2 = wrench(contact3_pos, f3_t2)
w4_n = wrench(contact4_pos, f4_n)
w4_t1 = wrench(contact4_pos, f4_t1)
w4_t2 = wrench(contact4_pos, f4_t2)



G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2, w4_n, w4_t1, w4_t2])


# Step 3: Compute Grasp Wrench Space vertices
vertices = []
for coeffs in np.eye(12):
    vertices.append(G @ coeffs)
for coeffs in [np.ones(12), -np.ones(12)]:
    vertices.append(G @ coeffs)
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Step 4: Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(12)

constraints = [
    w == G @ f,
    cp.norm(f, 1) <= 1,
    f >= 0
]

if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)

objective = cp.Maximize(r)
problem = cp.Problem(objective, constraints)
problem.solve()

quality = problem.value
print(f"Grasp Quality (Largest Minimum Wrench): {quality:.4f} N or Nm")
print(f"Center of largest inscribed ball: {w.value}")

# Step 5: Interactive 3D Plot with Plotly (Fx, Fy, Tz projection)
fig = go.Figure()

# GWS vertices
#fig.add_trace(go.Scatter3d(
 #   x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 5],
  #  mode='markers', marker=dict(size=5, color='blue'),
   # name='GWS Vertices'
#))

# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', marker=dict(size=8, color='red'),
    name='Origin'
))

# Center of the ball

fig.add_trace(go.Scatter3d(
    x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
    mode='markers', marker=dict(size=8, color='green'),
    name='Center'
))


# Quality sphere (parametric)
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x = quality * np.cos(u) * np.sin(v) + w.value[0]
y = quality * np.sin(u) * np.sin(v) + w.value[1]
z = quality * np.cos(v) + w.value[5]
fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    opacity=0.4,  # Slightly more visible but still see-through
    colorscale='Blues',
    showscale=False,
    name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
    surfacecolor=np.ones_like(x),  # Uniform red
    contours=dict(
        x=dict(show=True, color='skyblue', width=1),
        y=dict(show=True, color='skyblue', width=1),
        z=dict(show=True, color='skyblue', width=1)
    )  # Add wireframe-like contours
))

# Layout
fig.update_layout(
    title='Grasp Wrench Space (Fx-Fy-Tz Projection) / 4 Finger Symmteric',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

ft1  [-0.63245553 -0.63245553  0.4472136 ]
Grasp Quality (Largest Minimum Wrench): 0.2203 N or Nm
Center of largest inscribed ball: [-2.85775177e-11  3.19935612e-11  1.39378465e-01 -1.91244278e-11
 -1.90917451e-11 -7.60690815e-02]


In [ ]:
#2 Finger Symmetric / Contact line
# Function to compute wrench
def wrench(pos, force):
    return np.concatenate([force, np.cross(pos, force)])

# Friction coefficient
mu = 0.5

# --- Parallel Gripper with Line Contacts (4 contact points) ---
# Right finger: two points
contact1_pos = np.array([1, 0.05, 0])  # Right face, top
contact2_pos = np.array([1, -0.05, 0]) # Right face, bottom
# Left finger: two points
contact3_pos = np.array([-1, 0.05, 0]) # Left face, top
contact4_pos = np.array([-1, -0.05, 0])# Left face, bottom

# Normals (inward)
normal1 = np.array([-1, 0, 0])  # Right finger, top
normal2 = np.array([-1, 0, 0])  # Right finger, bottom
normal3 = np.array([1, 0, 0])   # Left finger, top
normal4 = np.array([1, 0, 0])   # Left finger, bottom

# Tangents (z and y directions for friction cone)
tangent1_a = np.array([0, 0, 1]); tangent1_b = np.array([0, 1, 0])
tangent2_a = np.array([0, 0, 1]); tangent2_b = np.array([0, 1, 0])
tangent3_a = np.array([0, 0, 1]); tangent3_b = np.array([0, 1, 0])
tangent4_a = np.array([0, 0, 1]); tangent4_b = np.array([0, 1, 0])

# Friction cone forces
f1_n = normal1; f1_t1 = normal1 + mu * tangent1_a; f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2; f2_t1 = normal2 + mu * tangent2_a; f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3; f3_t1 = normal3 + mu * tangent3_a; f3_t2 = normal3 + mu * tangent3_b
f4_n = normal4; f4_t1 = normal4 + mu * tangent4_a; f4_t2 = normal4 + mu * tangent4_b

# Normalize
for f in [f1_t1, f1_t2, f2_t1, f2_t2, f3_t1, f3_t2, f4_t1, f4_t2]:
    f /= np.linalg.norm(f)

# Wrenches
w1_n = wrench(contact1_pos, f1_n); w1_t1 = wrench(contact1_pos, f1_t1); w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n); w2_t1 = wrench(contact2_pos, f2_t1); w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n); w3_t1 = wrench(contact3_pos, f3_t1); w3_t2 = wrench(contact3_pos, f3_t2)
w4_n = wrench(contact4_pos, f4_n); w4_t1 = wrench(contact4_pos, f4_t1); w4_t2 = wrench(contact4_pos, f4_t2)

# Grasp matrix (6 x 12, 3 wrenches per contact, 4 contacts)
G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2, w4_n, w4_t1, w4_t2])
G_size = G.shape[1]
# Compute GWS vertices
vertices = [G @ coeffs for coeffs in np.eye(G_size)] + [G @ np.ones(G_size), G @ -np.ones(G_size)]
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(G_size)
constraints = [w == G @ f, cp.norm(f, 1) <= 1, f >= 0]
if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)
problem = cp.Problem(cp.Maximize(r), constraints)
problem.solve()
quality = problem.value if problem.status == cp.OPTIMAL else 0
print(f"Parallel Gripper Quality (Line Contacts): {quality:.4f} N or Nm (Status: {problem.status})")
if quality > 0:
    print(f"Center of largest inscribed ball: {w.value}")

# Interactive 3D Plot with Plotly (Fx-Fy-Tz projection)
fig = go.Figure()

# GWS vertices
#fig.add_trace(go.Scatter3d(
 #   x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 5],
  #  mode='markers',
  #  marker=dict(size=3, color='blue', opacity=0.6),
  #  name='GWS Vertices'
#))

# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=8, color='red'),
    name='Origin'
))

# Center and sphere (if quality > 0)
if quality > 0:
    fig.add_trace(go.Scatter3d(
        x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
        mode='markers',
        marker=dict(size=6, color='green'),
        name='Center'
    ))
    u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
    x = quality * np.cos(u) * np.sin(v) + w.value[0]
    y = quality * np.sin(u) * np.sin(v) + w.value[1]
    z = quality * np.cos(v) + w.value[5]
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        opacity=0.4,
        colorscale='Blues',
        showscale=False,
        name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
        contours=dict(
            x=dict(show=True, color='skyblue', width=1),
            y=dict(show=True, color='skyblue', width=1),
            z=dict(show=True, color='skyblue', width=1)
        )
    ))

# Layout
fig.update_layout(
    title='Parallel Gripper with Line Contacts (Fx-Fy-Tz) / Parallel Gripper (Line Contacts)',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

Parallel Gripper Quality (Line Contacts): 0.0214 N or Nm (Status: optimal)
Center of largest inscribed ball: [ 7.54836941e-10  1.22580281e-02  4.22183266e-01 -3.63463095e-15
  3.77418458e-10 -4.44679206e-12]


In [68]:
#3 Finger Antropomorphic

# Step 1: Define the gripper and object (3D)
contact1_pos = np.array([1, 1, 0])   # Finger 1 (right face)
contact2_pos = np.array([-1, 1, 0])  # Finger 2 (left face)
contact3_pos = np.array([0, -1, 0])   # Thumb (top face)
mu = 0.5  # Friction coefficient

# Step 2: Define wrench basis for each contact
normal1 = np.array([0, -1, 0])  # Right face, inward
normal2 = np.array([0, -1, 0])   # Left face, inward
normal3 = np.array([0, 1, 0])  # Top face, inward


# Contact 1 (Finger 1)
tangent1_a = np.array([0, 0, 1])
tangent1_b = np.array([1, 0, 0])

# Contact 2 (Finger 2)
tangent2_a = np.array([0, 0, 1])
tangent2_b = np.array([1, 0, 0])

# Contact 3 (Thumb)
tangent3_a = np.array([0, 0, 1])
tangent3_b = np.array([-1, 0, 0])

f1_n = normal1
f1_t1 = normal1 + mu * tangent1_a
f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2
f2_t1 = normal2 + mu * tangent2_a
f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3
f3_t1 = normal3 + mu * tangent3_a
f3_t2 = normal3 + mu * tangent3_b

f1_t1 /= np.linalg.norm(f1_t1)
f1_t2 /= np.linalg.norm(f1_t2)
f2_t1 /= np.linalg.norm(f2_t1)
f2_t2 /= np.linalg.norm(f2_t2)
f3_t1 /= np.linalg.norm(f3_t1)
f3_t2 /= np.linalg.norm(f3_t2)

def wrench(pos, force):
    r = pos
    f = force
    torque = np.cross(r, f)
    return np.concatenate([f, torque])

w1_n = wrench(contact1_pos, f1_n)
w1_t1 = wrench(contact1_pos, f1_t1)
w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n)
w2_t1 = wrench(contact2_pos, f2_t1)
w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n)
w3_t1 = wrench(contact3_pos, f3_t1)
w3_t2 = wrench(contact3_pos, f3_t2)



G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2])


# Step 3: Compute Grasp Wrench Space vertices
vertices = []
for coeffs in np.eye(9):
    vertices.append(G @ coeffs)
for coeffs in [np.ones(9), -np.ones(9)]:
    vertices.append(G @ coeffs)
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Step 4: Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(9)

constraints = [
    w == G @ f,
    cp.norm(f, 1) <= 1,
    f >= 0
]

if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)

objective = cp.Maximize(r)
problem = cp.Problem(objective, constraints)
problem.solve()

quality = problem.value
print(f"Grasp Quality (Largest Minimum Wrench): {quality:.4f} N or Nm")
print(f"Center of largest inscribed ball: {w.value}")

# Step 5: Interactive 3D Plot with Plotly (Fx, Fy, Tz projection)
fig = go.Figure()

# GWS vertices


# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', marker=dict(size=8, color='red'),
    name='Origin'
))

# Center of the ball

fig.add_trace(go.Scatter3d(
    x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
    mode='markers', marker=dict(size=8, color='green'),
    name='Center'
))


# Quality sphere (parametric)
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x = quality * np.cos(u) * np.sin(v) + w.value[0]
y = quality * np.sin(u) * np.sin(v) + w.value[1]
z = quality * np.cos(v) + w.value[5]
fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    opacity=0.4,  # Slightly more visible but still see-through
    colorscale='Reds',
    showscale=False,
    name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
    surfacecolor=np.ones_like(x),  # Uniform red
    contours=dict(
        x=dict(show=True, color='red', width=1),
        y=dict(show=True, color='red', width=1),
        z=dict(show=True, color='red', width=1)
    )  # Add wireframe-like contours
))

# Layout
fig.update_layout(
    title='Grasp Wrench Space (Fx-Fy-Tz Projection) / 3 Finger Antropomorphic',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

Grasp Quality (Largest Minimum Wrench): 0.1438 N or Nm
Center of largest inscribed ball: [ 3.29674420e-02 -2.59507343e-01  1.88590222e-01  5.78913895e-02
 -3.09047878e-10 -1.01687434e-01]


In [69]:
# 3 Finger Antropomorphic with contact line
# Function to compute wrench
def wrench(pos, force):
    return np.concatenate([force, np.cross(pos, force)])

mu = 0.8

# --- 3-Finger Gripper with Line Contacts (6 contact points) ---
# Finger 1: two points
contact1_pos = np.array([1, 1, 0.1])  # Right face, top
contact2_pos = np.array([1, 1, -0.1]) # Right face, bottom
# Finger 2: two points
contact3_pos = np.array([-1, 1, 0.1]) # Left face, top
contact4_pos = np.array([-1, 1, -0.1])# Left face, bottom
# Thumb: two points
contact5_pos = np.array([0, -1, 0.1]) # Thumb, top
contact6_pos = np.array([0, -1, -0.1])# Thumb, bottom

# Normals (inward)
normal1 = np.array([0, -1, 0])  # Finger 1, top
normal2 = np.array([0, -1, 0])  # Finger 1, bottom
normal3 = np.array([0, -1, 0])  # Finger 2, top
normal4 = np.array([0, -1, 0])  # Finger 2, bottom
normal5 = np.array([0, 1, 0])   # Thumb, top
normal6 = np.array([0, 1, 0])   # Thumb, bottom

# Tangents
tangent1_a = np.array([0, 0, 1]); tangent1_b = np.array([1, 0, 0])
tangent2_a = np.array([0, 0, 1]); tangent2_b = np.array([1, 0, 0])
tangent3_a = np.array([0, 0, 1]); tangent3_b = np.array([1, 0, 0])
tangent4_a = np.array([0, 0, 1]); tangent4_b = np.array([1, 0, 0])
tangent5_a = np.array([0, 0, 1]); tangent5_b = np.array([-1, 0, 0])
tangent6_a = np.array([0, 0, 1]); tangent6_b = np.array([-1, 0, 0])

# Friction cone forces
f1_n = normal1; f1_t1 = normal1 + mu * tangent1_a; f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2; f2_t1 = normal2 + mu * tangent2_a; f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3; f3_t1 = normal3 + mu * tangent3_a; f3_t2 = normal3 + mu * tangent3_b
f4_n = normal4; f4_t1 = normal4 + mu * tangent4_a; f4_t2 = normal4 + mu * tangent4_b
f5_n = normal5; f5_t1 = normal5 + mu * tangent5_a; f5_t2 = normal5 + mu * tangent5_b
f6_n = normal6; f6_t1 = normal6 + mu * tangent6_a; f6_t2 = normal6 + mu * tangent6_b

# Normalize
for f in [f1_t1, f1_t2, f2_t1, f2_t2, f3_t1, f3_t2, f4_t1, f4_t2, f5_t1, f5_t2, f6_t1, f6_t2]:
    f /= np.linalg.norm(f)

# Wrenches
w1_n = wrench(contact1_pos, f1_n); w1_t1 = wrench(contact1_pos, f1_t1); w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n); w2_t1 = wrench(contact2_pos, f2_t1); w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n); w3_t1 = wrench(contact3_pos, f3_t1); w3_t2 = wrench(contact3_pos, f3_t2)
w4_n = wrench(contact4_pos, f4_n); w4_t1 = wrench(contact4_pos, f4_t1); w4_t2 = wrench(contact4_pos, f4_t2)
w5_n = wrench(contact5_pos, f5_n); w5_t1 = wrench(contact5_pos, f5_t1); w5_t2 = wrench(contact5_pos, f5_t2)
w6_n = wrench(contact6_pos, f6_n); w6_t1 = wrench(contact6_pos, f6_t1); w6_t2 = wrench(contact6_pos, f6_t2)

# Grasp matrix (6 x 18, 3 wrenches per contact, 6 contacts)
G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2,
                     w4_n, w4_t1, w4_t2, w5_n, w5_t1, w5_t2, w6_n, w6_t1, w6_t2])

G_shape_cols = G.shape[1]
# Compute GWS vertices
vertices = [G @ coeffs for coeffs in np.eye(G_shape_cols)] + [G @ np.ones(G_shape_cols), G @ -np.ones(G_shape_cols)]
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(G_shape_cols)
constraints = [w == G @ f, cp.norm(f, 1) <= 1, f >= 0]
if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)
problem = cp.Problem(cp.Maximize(r), constraints)
problem.solve()
quality = problem.value if problem.status == cp.OPTIMAL else 0
print(f"3-Finger Gripper Quality (Line Contacts): {quality:.4f} N or Nm (Status: {problem.status})")
if quality > 0:
    print(f"Center of largest inscribed ball: {w.value}")

# Interactive 3D Plot with Plotly (Fx-Fy-Tz projection)
fig = go.Figure()



# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=8, color='black'),
    name='Origin'
))

# Center and sphere (if quality > 0)
if quality > 0:
    fig.add_trace(go.Scatter3d(
        x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
        mode='markers',
        marker=dict(size=6, color='green'),
        name='Center'
    ))
    u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
    x = quality * np.cos(u) * np.sin(v) + w.value[0]
    y = quality * np.sin(u) * np.sin(v) + w.value[1]
    z = quality * np.cos(v) + w.value[5]
    fig.add_trace(go.Surface(
        x=x, y=y, z=z,
        opacity=0.4,
        colorscale='Blues',
        showscale=False,
        name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
        contours=dict(
            x=dict(show=True, color='powderblue', width=1),
            y=dict(show=True, color='powderblue', width=1),
            z=dict(show=True, color='powderblue', width=1)
        )
    ))

# Layout
fig.update_layout(
    title='3-Finger Gripper with Line Contacts (Fx-Fy-Tz)',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

3-Finger Gripper Quality (Line Contacts): 0.2528 N or Nm (Status: optimal)
Center of largest inscribed ball: [ 0.08129577 -0.48468847  0.29819307  0.14921657 -0.01602615 -0.27405615]


In [71]:
#5 Finger Antropomorphic 

# Step 1: Define the gripper and object (3D)
contact1_pos = np.array([1, 1, 0])   # Finger 1 (right face)
contact2_pos = np.array([-1, 1, 0])  # Finger 2 (left face)
contact3_pos = np.array([0, -1, 0])   # Thumb (top face)

contact4_pos = np.array([0, 1, 0]) 
contact5_pos = np.array([-1, 0, 0]) 

mu = 0.5  # Friction coefficient

alpha = np.pi/6
beta = np.pi/2 - alpha
# Step 2: Define wrench basis for each contact
normal1 = np.array([-1 * np.cos(alpha), -1 * np.sin(alpha), 0])  # Right face, inward
normal2 = np.array([1 * np.cos(alpha), -1 * np.cos(alpha), 0]) 
normal3 = np.array([0, 1, 0])  # Top face, inward

normal4 = np.array([0, -1, 0])  # Top face, inward
normal5 = np.array([1, 0, 0])  # Top face, inward


# Contact 1 
tangent1_a = np.array([0, 0, 1])
tangent1_b = np.array([1 * np.cos(beta), -1 * np.sin(beta), 0])

# Contact 2 
tangent2_a = np.array([0, 0, 1])
tangent2_b = np.array([1 * np.cos(beta), 1 * np.sin(beta), 0])

# Contact 3 (Thumb)
tangent3_a = np.array([0, 0, 1])
tangent3_b = np.array([-1, 0, 0])

# Contact 4
tangent4_a = np.array([0, 0, 1])
tangent4_b = np.array([1, 0, 0])
# Contact 5
tangent5_a = np.array([0, 0, 1])
tangent5_b = np.array([0, 1, 0])

f1_n = normal1
f1_t1 = normal1 + mu * tangent1_a
f1_t2 = normal1 + mu * tangent1_b
f2_n = normal2
f2_t1 = normal2 + mu * tangent2_a
f2_t2 = normal2 + mu * tangent2_b
f3_n = normal3
f3_t1 = normal3 + mu * tangent3_a
f3_t2 = normal3 + mu * tangent3_b

f4_n = normal4
f4_t1 = normal4 + mu * tangent4_a
f4_t2 = normal4 + mu * tangent4_b

f5_n = normal5
f5_t1 = normal5 + mu * tangent5_a
f5_t2 = normal5 + mu * tangent5_b

f1_t1 /= np.linalg.norm(f1_t1)
f1_t2 /= np.linalg.norm(f1_t2)
f2_t1 /= np.linalg.norm(f2_t1)
f2_t2 /= np.linalg.norm(f2_t2)
f3_t1 /= np.linalg.norm(f3_t1)
f3_t2 /= np.linalg.norm(f3_t2)
f4_t1 /= np.linalg.norm(f4_t1)
f4_t2 /= np.linalg.norm(f4_t2)
f5_t1 /= np.linalg.norm(f5_t1)
f5_t2 /= np.linalg.norm(f5_t2)

def wrench(pos, force):
    r = pos
    f = force
    torque = np.cross(r, f)
    return np.concatenate([f, torque])

w1_n = wrench(contact1_pos, f1_n)
w1_t1 = wrench(contact1_pos, f1_t1)
w1_t2 = wrench(contact1_pos, f1_t2)
w2_n = wrench(contact2_pos, f2_n)
w2_t1 = wrench(contact2_pos, f2_t1)
w2_t2 = wrench(contact2_pos, f2_t2)
w3_n = wrench(contact3_pos, f3_n)
w3_t1 = wrench(contact3_pos, f3_t1)
w3_t2 = wrench(contact3_pos, f3_t2)

w4_n = wrench(contact4_pos, f4_n)
w4_t1 = wrench(contact4_pos, f4_t1)
w4_t2 = wrench(contact4_pos, f4_t2)
w5_n = wrench(contact5_pos, f5_n)
w5_t1 = wrench(contact5_pos, f5_t1)
w5_t2 = wrench(contact5_pos, f5_t2)



G = np.column_stack([w1_n, w1_t1, w1_t2, w2_n, w2_t1, w2_t2, w3_n, w3_t1, w3_t2, w4_n, w4_t1,w4_t2, w5_n, w5_t1, w5_t2])

G_shape = G.shape[1]

# Step 3: Compute Grasp Wrench Space vertices
vertices = []
for coeffs in np.eye(G_shape):
    vertices.append(G @ coeffs)
for coeffs in [np.ones(G_shape), -np.ones(G_shape)]:
    vertices.append(G @ coeffs)
vertices = np.array(vertices)

try:
    hull = ConvexHull(vertices)
except:
    print("Convex hull computation failed. Using vertices as-is.")
    hull = None

# Step 4: Quality metric - Largest inscribed ball
w = cp.Variable(6)
r = cp.Variable()
f = cp.Variable(G_shape)

constraints = [
    w == G @ f,
    cp.norm(f, 1) <= 1,
    f >= 0
]

if hull:
    for simplex in hull.equations:
        normal, offset = simplex[:-1], simplex[-1]
        constraints.append(cp.sum(normal @ w) + r * cp.norm(normal, 2) <= -offset)

objective = cp.Maximize(r)
problem = cp.Problem(objective, constraints)
problem.solve()

quality = problem.value
print(f"Grasp Quality (Largest Minimum Wrench): {quality:.4f} N or Nm")
print(f"Center of largest inscribed ball: {w.value}")

# Step 5: Interactive 3D Plot with Plotly (Fx, Fy, Tz projection)
fig = go.Figure()

# GWS vertices


# Origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', marker=dict(size=8, color='red'),
    name='Origin'
))

# Center of the ball

fig.add_trace(go.Scatter3d(
    x=[w.value[0]], y=[w.value[1]], z=[w.value[5]],
    mode='markers', marker=dict(size=8, color='green'),
    name='Center'
))


# Quality sphere (parametric)
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x = quality * np.cos(u) * np.sin(v) + w.value[0]
y = quality * np.sin(u) * np.sin(v) + w.value[1]
z = quality * np.cos(v) + w.value[5]
fig.add_trace(go.Surface(
    x=x, y=y, z=z,
    opacity=0.4,  # Slightly more visible but still see-through
    colorscale='Blues',
    showscale=False,
    name=f'Largest Minimum Wrench ({quality:.4f} N or Nm)',
    surfacecolor=np.ones_like(x),  # Uniform red
    contours=dict(
        x=dict(show=True, color='powderblue', width=1),
        y=dict(show=True, color='powderblue', width=1),
        z=dict(show=True, color='powderblue', width=1)
    )  # Add wireframe-like contours
))

# Layout
fig.update_layout(
    title='Grasp Wrench Space (Fx-Fy-Tz Projection) / 5 Finger Antropomorphic',
    scene=dict(
        xaxis_title='Fx (N)',
        yaxis_title='Fy (N)',
        zaxis_title='Tz (Nm)',
        aspectmode='cube'
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

Grasp Quality (Largest Minimum Wrench): 0.1779 N or Nm
Center of largest inscribed ball: [ 0.2397054  -0.1359782   0.22042908  0.04907999  0.05855993 -0.0591781 ]
